In [31]:
# Core imports
import os
import json
import requests
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✅ Setup complete")

✅ Setup complete


In [32]:
import ollama

# Test simple prompt
response = ollama.chat(
    model="llama3.2",
    messages=[
        {"role": "user", "content": "Say hello in one sentence"}
    ]
)

print(response["message"]["content"])

Hello!


In [ ]:
NAME = "Nikith"  # change to your name

SYSTEM_PROMPT = f"""
You are acting as {NAME}, a professional AI/Cloud Engineer.

You must answer like {NAME}, based on real experience, skills, and background.

RULES:
- Answer as if you are {NAME}
- Use first-person ("I", "my experience")
- Base answers on provided LinkedIn + summary
- Be professional but natural
- If unknown, say you don’t know

---

### SUMMARY:
{summary_text}

---

### LINKEDIN PROFILE:
{linkedin_text}
"""

In [34]:
def chat_with_llama(user_input, chat_history=[]):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ] + chat_history + [
        {"role": "user", "content": user_input}
    ]

    response = ollama.chat(
        model="llama3.2",
        messages=messages
    )

    reply = response["message"]["content"]

    # Update history
    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": reply})

    return reply, chat_history

In [35]:
chat_history = []

reply, chat_history = chat_with_llama("Explain AI in 2 lines", chat_history)
print(reply)

Artificial Intelligence (AI) refers to computer systems that can think, learn, and act like humans. It enables machines to process data, make decisions, and perform tasks autonomously with increasing accuracy and efficiency.


In [36]:
GENERATOR_MODEL = "llama3.2"
EVALUATOR_MODEL = "phi3"

In [45]:
def clean_response(text):
    parts = text.strip().split("\n\n")
    seen = []
    for p in parts:
        if p not in seen:
            seen.append(p)
    return "\n\n".join(seen)

In [46]:
def generator_agent(user_input, chat_history):
    user_input = user_input + "\n\nFollow the instruction strictly. Keep it short."

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ] + chat_history + [
        {"role": "user", "content": user_input}
    ]

    response = ollama.chat(
        model=GENERATOR_MODEL,
        messages=messages
    )

    return response["message"]["content"]

In [47]:
EVALUATOR_PROMPT = """
You are a STRICT AI evaluator.

Return ONLY valid JSON.
DO NOT add any explanation or text outside JSON.

Format:
{
  "score": 1-10,
  "verdict": "approve" or "improve",
  "feedback": "short feedback",
  "improved_response": "corrected response"
}
"""

In [48]:
def evaluator_agent(user_input, response):
    short_response = response[:300]

    messages = [
        {"role": "system", "content": EVALUATOR_PROMPT},
        {"role": "user", "content": f"""
Question: {user_input}

Response: {short_response}
"""}
    ]

    eval_response = ollama.chat(
        model=EVALUATOR_MODEL,
        messages=messages
    )

    content = eval_response["message"]["content"]

    # Extract JSON safely
    try:
        json_str = re.search(r"\{.*\}", content, re.DOTALL).group()
        return json_str
    except:
        return '{"score": 5, "verdict": "approve", "feedback": "fallback", "improved_response": ""}'

In [52]:
def multi_agent_chat(user_input, chat_history=[]):
    # Step 1: Generate
    raw_response = generator_agent(user_input, chat_history)

    # Step 2: Evaluate
    evaluation = evaluator_agent(user_input, raw_response)

    try:
        eval_json = json.loads(evaluation)
    except:
        eval_json = {
            "score": "N/A",
            "verdict": "approve",
            "feedback": "JSON parsing failed",
            "improved_response": raw_response
        }

    # Step 3: Decision Logic (SMART)
    improved = eval_json.get("improved_response", "").strip()

    # Case 1: Force improvement if too long
    if len(raw_response.split()) > 80:
        final_response = improved if improved and improved != raw_response else raw_response

    # Case 2: If evaluator suggests improvement → use it
    elif improved:
        final_response = improved

    # Case 3: Otherwise keep original
    else:
        final_response = raw_response

    # Step 4: Clean duplicates
    final_response = clean_response(final_response)

    # Step 5: Update history
    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": final_response})

    # Step 6: Debug output
    print("\n🧠 Generator Response:\n", raw_response)
    print("\n🔍 Evaluator Output:\n", eval_json)
    print("\n✅ Final Response:\n", final_response)

    return final_response, chat_history

In [53]:
chat_history = []

reply, chat_history = multi_agent_chat(
    "Explain AI in 2 lines for a beginner",
    chat_history
)

print("\n🎯 OUTPUT:\n", reply)


🧠 Generator Response:
 Artificial Intelligence (AI) is a computer system that can think and learn like humans. It uses complex algorithms to analyze data, make decisions, and improve its performance over time.

🔍 Evaluator Output:
 {'score': 8, 'verdict': 'improve', 'feedback': 'The response provides a basic understanding of AI but lacks simplicity for beginners.', 'improved_response': 'AI is like an electronic brain that can learn and get smarter. It makes choices by studying information.'}

✅ Final Response:
 AI is like an electronic brain that can learn and get smarter. It makes choices by studying information.

🎯 OUTPUT:
 AI is like an electronic brain that can learn and get smarter. It makes choices by studying information.
